# 02 — Per-site K_d retrieval and bootstrap

This is the **main computation** of the paper. It runs the per-site
K_d sweep at Apollo 15 and 17 and the non-parametric bootstrap
(N_boot = 1500, sensor-placement uncertainty +-2.5 cm).

Wall time: ~20-40 min on a recent laptop (depending on CPU cores).

Output: [`output/phase_a_results.json`](../output/phase_a_results.json)

Once this notebook finishes, notebooks 03 and 04 can be re-run
independently to retune figures without redoing the heavy lifting.

In [ ]:
import sys, pathlib, subprocess
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

## Phase A — central retrieval pipeline

This runs `scripts/pipeline/phase_a_pipeline.py` which:
1. Loads the bundled HFE record
2. Defines the K_d sweep grid for each site
3. Runs the 1-D Crank-Nicolson solver at each grid point
4. Identifies the RMSE minimum
5. Runs the bootstrap (1500 resamples, +-2.5 cm depth jitter)
6. Writes the canonical results to `output/phase_a_results.json`

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'pipeline' / 'phase_a_pipeline.py')],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout[-2000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError(f'phase_a_pipeline failed (exit {result.returncode})')
print('Phase A complete.')

## Headline result

In [ ]:
import json
d = json.loads((ROOT / 'output' / 'phase_a_results.json').read_text())
for site in ('A15', 'A17'):
    s = d[site]
    b = s['bootstrap']
    print(f'  {site}:')
    print(f'    K_d*        = {s["kd_star"]*1e3:.3f} mW m^-1 K^-1')
    print(f'    RMSE at K_d*= {s["rmse_star"]:.3f} K')
    print(f'    bootstrap median = {b["median"]*1e3:.2f} mW m^-1 K^-1')
    print(f'    95% CI      = [{b["ci_lo"]*1e3:.2f}, {b["ci_hi"]*1e3:.2f}] mW m^-1 K^-1')
    print()

## Auxiliary sensitivity sweeps

These produce the supporting JSONs read by the error-budget table:
- `borestem_sensitivity.json` — z_b cut sweep (60, 70, 80, 90 cm)
- `stability_threshold_sensitivity.json` — stability-window threshold sweep
- `surface_bias_test.json` — Bond-albedo perturbation
- `headline_rmse.json` — comparison vs published global K_d and Martinez forward
- `model_selection.json` — AICc model comparison
- `kd_qb_envelope.json` — K_d / Q_b degeneracy mapping

In [ ]:
for script in [
    'compute_borestem_sensitivity.py',
    'compute_stability_threshold_sensitivity.py',
    'compute_surface_bias_test.py',
    'compute_uniform_kd_test.py',
    'compute_headline_rmse.py',
    'compute_model_selection.py',
]:
    print(f'\n=== {script} ===')
    r = subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'pipeline' / script)],
        cwd=str(ROOT), capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('FAILED:', r.stderr[-500:])
    else:
        print(r.stdout[-500:] if r.stdout else '(no output)')

## Phase B — Bayesian cross-check (MCMC)

Independent cross-check of the contrast direction (not magnitude).
Skip this cell if you don't need the Bayesian section.

In [ ]:
r = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'pipeline' / 'phase_b_mcmc.py')],
    cwd=str(ROOT), capture_output=True, text=True,
)
print(r.stdout[-1000:] if r.stdout else '')
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    print('Phase B failed -- skipping (not required for the headline result).')
else:
    print('Phase B complete.')